<a href="https://colab.research.google.com/github/rxnu/LLM-Project/blob/main/4_optimization_and_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate


In [ ]:

# reload and split IMDB
ds = load_dataset("imdb")
ds = DatasetDict({
    "train":      ds["train"].shuffle(seed=42).select(range(22500)),
    "validation": ds["train"].shuffle(seed=42).select(range(22500, 25000)),
    "test":       ds["test"],
})

# load baseline
model_name = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)

## reuse ds, compute_metrics from notebook 3
# tokenize
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

ds = ds.map(tokenize_batch, batched=True)

# keep only the columns Trainer needs
to_keep = ["input_ids", "attention_mask", "label"]
for split in ["train", "validation", "test"]:
    ds[split] = ds[split].remove_columns(
        [c for c in ds[split].column_names if c not in to_keep]
    )

# rename & format for PyTorch
ds = ds.rename_column("label", "labels")
ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# prepare metrics
accuracy = evaluate.load("accuracy")
f1       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1":       f1.compute(predictions=preds, references=labels)["f1"],
    }


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
# reload saved baseline from Hugging Face Hub
model_name = "rxnu/imdb-distilbert-finetuned"

model     = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
repo_name = "rxnu/imdb-distilbert-optimized"

# Experiment v1
training_args = TrainingArguments(
    output_dir=repo_name,
    push_to_hub=True,             # automatically push at end
    learning_rate=3e-5,           # slightly higher LR
    num_train_epochs=4,           # one more epoch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.0,             # remove weight decay
    warmup_steps=500,             # add warmup
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# train & evaluate
trainer.train()
print("Validation:", trainer.evaluate(ds["validation"]))
print("Test set:",   trainer.evaluate(ds["test"]))


/tmp/ipython-input-8-454599398.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.114500,0.418544,0.892000,0.891304
2,0.079600,0.488658,0.897600,0.897436
3,0.055400,0.462535,0.909200,0.913721
4,0.021800,0.527207,0.908800,0.911696


Validation: {'eval_loss': 0.46253514289855957, 'eval_accuracy': 0.9092, 'eval_f1': 0.9137210186240973, 'eval_runtime': 16.7722, 'eval_samples_per_second': 149.056, 'eval_steps_per_second': 4.71, 'epoch': 4.0}
Test set: {'eval_loss': 0.4681963622570038, 'eval_accuracy': 0.90656, 'eval_f1': 0.9080314960629922, 'eval_runtime': 167.0074, 'eval_samples_per_second': 149.694, 'eval_steps_per_second': 4.682, 'epoch': 4.0}


In [ ]:
trainer.push_to_hub()

Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/rxnu/imdb-distilbert-optimized/commit/32c6c2afb4bd6a16f97ed2ae784d606810f248e2', commit_message='End of training', commit_description='', oid='32c6c2afb4bd6a16f97ed2ae784d606810f248e2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rxnu/imdb-distilbert-optimized', endpoint='https://huggingface.co', repo_type='model', repo_id='rxnu/imdb-distilbert-optimized'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Define Drive path & save there
drive_path = "/content/drive/MyDrive/imdb-distilbert-optimized"
trainer.save_model(drive_path)                # saves model weights + config
tokenizer.save_pretrained(drive_path)         # saves vocab/tokenizer files

print(f"Pushed to Hub and also saved to Drive at: {drive_path}")

Mounted at /content/drive


Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

✅ Pushed to Hub and also saved to Drive at: /content/drive/MyDrive/imdb-distilbert-optimized


In [ ]:
from transformers import EarlyStoppingCallback

# Experiment v2
repo_name = "rxnu/imdb-distilbert-optimized-v2"

training_args = TrainingArguments(
    output_dir=repo_name,
    push_to_hub=True,
    learning_rate=2e-5,               # baseline LR
    weight_decay=0.01,                # re-add weight decay
    warmup_ratio=0.1,                 # 10% of training steps
    num_train_epochs=3,               # back to 3 epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,    # bigger eval batch
    gradient_accumulation_steps=2,    # effective train batch 32
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)

# instantiate Trainer with early stopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()

#final eval
print("Validation:", trainer.evaluate(ds["validation"]))
print("Test set:  ", trainer.evaluate(ds["test"]))

# push to HF Hub and save to Drive
trainer.push_to_hub()

from google.colab import drive
drive.mount("/content/drive")

drive_path = "/content/drive/MyDrive/imdb-distilbert-optimized-v2"
trainer.save_model(drive_path)
tokenizer.save_pretrained(drive_path)
print("Saved optimized-v2 model to Drive at:", drive_path)

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.070900,0.330199,0.912800,0.916283
2,0.051500,0.400688,0.909600,0.911719


Validation: {'eval_loss': 0.3301994502544403, 'eval_accuracy': 0.9128, 'eval_f1': 0.9162826420890937, 'eval_runtime': 16.5196, 'eval_samples_per_second': 151.336, 'eval_steps_per_second': 2.421, 'epoch': 2.0}
Test set:   {'eval_loss': 0.3454495370388031, 'eval_accuracy': 0.90912, 'eval_f1': 0.909977018781203, 'eval_runtime': 166.1774, 'eval_samples_per_second': 150.442, 'eval_steps_per_second': 2.353, 'epoch': 2.0}


Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Saved optimized-v2 model to Drive at: /content/drive/MyDrive/imdb-distilbert-optimized-v2


In [ ]:
#Experiment v3 - set up a stronger TrainingArguments

repo_name = "rxnu/imdb-distilbert-optimized-v3"
training_args = TrainingArguments(
    output_dir=repo_name,
    push_to_hub=True,
    learning_rate=1e-5,              # lower LR
    weight_decay=0.02,               # slightly higher WD
    warmup_ratio=0.2,                # 20% warmup
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to=None,
    run_name=None,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()
print("Validation:", trainer.evaluate(ds["validation"]))
print("Test set:  ", trainer.evaluate(ds["test"]))
trainer.push_to_hub()

# Save to Drive too
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
drive_path = "/content/drive/MyDrive/imdb-distilbert-optimized-v3"
trainer.save_model(drive_path)
tokenizer.save_pretrained(drive_path)
print("Saved v3 to Drive:", drive_path)

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.055400,0.336825,0.906400,0.909722


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.055400,0.336825,0.906400,0.909722
2,0.046400,0.384024,0.913600,0.916279
3,0.029500,0.420121,0.913200,0.917144


Validation: {'eval_loss': 0.38402363657951355, 'eval_accuracy': 0.9136, 'eval_f1': 0.9162790697674419, 'eval_runtime': 16.5683, 'eval_samples_per_second': 150.89, 'eval_steps_per_second': 2.414, 'epoch': 3.0}
Test set:   {'eval_loss': 0.4042157530784607, 'eval_accuracy': 0.90896, 'eval_f1': 0.9086163976551835, 'eval_runtime': 166.4805, 'eval_samples_per_second': 150.168, 'eval_steps_per_second': 2.349, 'epoch': 3.0}


Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

Mounted at /content/drive


Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Saved v3 to Drive: /content/drive/MyDrive/imdb-distilbert-optimized-v3


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)

drive_path = "/content/drive/MyDrive/imdb-distilbert-optimized-v3"
print(os.path.isdir(drive_path), os.listdir(drive_path))

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(drive_path, local_files_only=True)
model     = AutoModelForSequenceClassification.from_pretrained(drive_path, local_files_only=True)



Mounted at /content/drive
True ['config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json']


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# re‐evaluate on the test split
from transformers import Trainer
import evaluate

test_out = Trainer(model=model, tokenizer=tokenizer).predict(ds["test"])
preds    = test_out.predictions.argmax(axis=-1)
labels   = test_out.label_ids

acc = evaluate.load("accuracy").compute(
    predictions=preds,
    references=labels
)["accuracy"]
f1  = evaluate.load("f1").compute(
    predictions=preds,
    references=labels
)["f1"]

print(f"→ v3 Test Accuracy: {acc:.4f}")
print(f"→ v3 Test   F1  : {f1:.4f}")

# quick inference demo on a few examples
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
)

examples = [
    "Absolutely loved this movie—one of the best I've seen!",
    "Complete waste of time, I walked out halfway through.",
    "It was okay: some parts were fun, others felt slow."
]

for ex, res in zip(examples, sentiment(examples)):
    print(f"> {ex}\n→ {res}\n")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Device set to use cuda:0


→ v3 Test Accuracy: 0.9090
→ v3 Test   F1  : 0.9086
> Absolutely loved this movie—one of the best I've seen!
→ {'label': 'LABEL_1', 'score': 0.9988530874252319}

> Complete waste of time, I walked out halfway through.
→ {'label': 'LABEL_0', 'score': 0.9983385801315308}

> It was okay: some parts were fun, others felt slow.
→ {'label': 'LABEL_0', 'score': 0.8679417967796326}

